In [ ]:
#Emprical method

In [7]:
import pandas as pd
import numpy as np

# =========================
# User inputs
# =========================
file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
#file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_XY_Final_Fixed.xlsx"
output_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated_with_rescaled_yields.xlsx"

# Fixed old plant count
N_OLD = 150

# Empirical exponent for the empirical rescaling method
ALPHA = 0.85   # change this if you want, e.g. 0.5, 1.0, 1.2
#ALPHA = 0.91   # change this if you want, e.g. 0.5, 1.0, 1.2

# Optional biomass proxy column:
# Put the name of your biomass proxy column here if you have one
# Examples: "CanopyArea", "NDVI_ActiveArea", "LiDAR_Volume", "GreenPixelArea"
# If None, the script will use HA / SHAPE_Area as a fallback biomass fraction proxy
BIOMASS_PROXY_COL = None

# =========================
# Read data
# =========================
df = pd.read_excel(file_path)

# =========================
# Check required columns
# =========================
required_cols = ["PLOTWT", "Uncorrected_Yield", "SHAPE_Area", "HA", "Join_Count"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in Excel file: {missing_cols}")

# =========================
# Clean numeric columns
# =========================
for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert observed plot weight from grams to kilograms
df["PLOTWT_kg"] = df["PLOTWT"] / 1000.0

# Convenience variables
W_obs = df["PLOTWT_kg"]              # observed production mass in kg
Y_old = df["Uncorrected_Yield"]      # original full-plot yield in kg/ha
A_old = df["SHAPE_Area"]             # original full plot area in m²
A_new = df["HA"]                     # new harvestable area in m²
N_new = df["Join_Count"]             # observed stand count / plant density proxy

# Valid rows
valid = (
    W_obs.notna() & Y_old.notna() &
    A_old.notna() & A_new.notna() & N_new.notna() &
    (A_old > 0) & (A_new > 0)
)

# ============================================================
# 1) Production-preserving rescaling
# Keep observed production mass constant, divide by new area
# Y_new = W_obs / A_new * 10000
# ============================================================
df["Yield_ProdPres_kg_ha"] = np.where(
    valid & (W_obs > 0),
    (W_obs / A_new) * 10000.0,
    np.nan
)

# ============================================================
# 2) Area-proportional biomass rescaling
# Scale biomass by area ratio, then convert back to kg/ha
# W_old_from_Y = Y_old * A_old / 10000
# W_new = W_old_from_Y * (A_new / A_old)
# Y_new = W_new / A_new * 10000
# This simplifies to Y_new = Y_old
# ============================================================
W_old_from_Y = (Y_old * A_old) / 10000.0

df["Yield_AreaPropBiomass_kg_ha"] = np.where(
    valid,
    ((W_old_from_Y * (A_new / A_old)) / A_new) * 10000.0,
    np.nan
)

# ============================================================
# 3) Productive-area rescaling
# Treat HA as the effective productive area and normalize
# by that area using the observed production mass
# With your current inputs this will match Production-preserving
# ============================================================
df["Yield_ProductiveArea_kg_ha"] = np.where(
    valid & (W_obs > 0),
    (W_obs / A_new) * 10000.0,
    np.nan
)

# ============================================================
# 4) Biomass-fraction rescaling
# W_new = W_old_from_Y * (B_new / B_old)
# Y_new = W_new / A_new * 10000
#
# If you provide a biomass proxy column:
#   B_old = max biomass proxy for full plot
#   B_new = observed biomass proxy in retained area
#
# Fallback:
#   B_new / B_old = A_new / A_old
# which makes this method collapse toward the area-proportional method
# ============================================================
if BIOMASS_PROXY_COL is not None and BIOMASS_PROXY_COL in df.columns:
    df[BIOMASS_PROXY_COL] = pd.to_numeric(df[BIOMASS_PROXY_COL], errors="coerce")

    # Here we assume the provided biomass proxy already represents the fraction
    # or observed biomass in the new area. We estimate B_old from full area.
    # If your biomass proxy is already a fraction, adapt this part as needed.
    B_new = df[BIOMASS_PROXY_COL]

    # A generic fallback assumption:
    # biomass of full plot scales with area if no explicit full-plot biomass column exists
    B_old = B_new * (A_old / A_new)

    biomass_valid = valid & B_new.notna() & B_old.notna() & (B_old > 0)

    df["Yield_BiomassFraction_kg_ha"] = np.where(
        biomass_valid,
        ((W_old_from_Y * (B_new / B_old)) / A_new) * 10000.0,
        np.nan
    )
else:
    # Fallback: use harvestable-area fraction as biomass fraction proxy
    biomass_fraction = A_new / A_old

    df["Yield_BiomassFraction_kg_ha"] = np.where(
        valid,
        ((W_old_from_Y * biomass_fraction) / A_new) * 10000.0,
        np.nan
    )

# ============================================================
# 5) Stand-count rescaling
# W_new = W_old_from_Y * (N_new / N_OLD)
# Y_new = W_new / A_new * 10000
# ============================================================
df["Yield_StandCount_kg_ha"] = np.where(
    valid & (N_new >= 0),
    ((W_old_from_Y * (N_new / N_OLD)) / A_new) * 10000.0,
    np.nan
)

# ============================================================
# 6) Empirical rescaling with exponent
# W_new = W_old_from_Y * (A_new / A_old)^alpha
# Y_new = W_new / A_new * 10000
# ============================================================
df["Yield_EmpiricalAlpha_kg_ha"] = np.where(
    valid,
    ((W_old_from_Y * ((A_new / A_old) ** ALPHA)) / A_new) * 10000.0,
    np.nan
)

# ============================================================
# Optional: add a few helper columns so you can inspect the behavior
# ============================================================
df["Area_Ratio_New_to_Old"] = np.where(valid, A_new / A_old, np.nan)
df["Stand_Ratio_New_to_Old"] = np.where(valid, N_new / N_OLD, np.nan)
df["Mass_From_UncorrectedYield_kg"] = np.where(valid, W_old_from_Y, np.nan)

# =========================
# Save output
# =========================
df.to_excel(output_path, index=False)

print("Done.")
print(f"Output saved to:\n{output_path}")

Done.
Output saved to:
C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated_with_rescaled_yields.xlsx


In [ ]:
#Global alpha

In [6]:
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LinearRegression

# # =========================
# # File path
# # =========================
# file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
# output_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_alpha.xlsx"

# # =========================
# # Read file
# # =========================
# df = pd.read_excel(file_path)

# # =========================
# # Required columns
# # =========================
# required_cols = ["Uncorrected_Yield", "Corrected_Yield", "SHAPE_Area", "HA"]
# missing_cols = [c for c in required_cols if c not in df.columns]
# if missing_cols:
#     raise ValueError(f"Missing columns: {missing_cols}")

# # Convert to numeric
# for col in required_cols:
#     df[col] = pd.to_numeric(df[col], errors="coerce")

# # =========================
# # Valid rows for alpha estimation
# # =========================
# valid = (
#     df["Uncorrected_Yield"].notna() &
#     df["Corrected_Yield"].notna() &
#     df["SHAPE_Area"].notna() &
#     df["HA"].notna() &
#     (df["Uncorrected_Yield"] > 0) &
#     (df["Corrected_Yield"] > 0) &
#     (df["SHAPE_Area"] > 0) &
#     (df["HA"] > 0) &
#     (df["HA"] != df["SHAPE_Area"])
# )

# # =========================
# # Row-wise alpha
# # =========================
# df["Alpha_Row"] = np.nan

# df.loc[valid, "Alpha_Row"] = (
#     1
#     + np.log(df.loc[valid, "Corrected_Yield"] / df.loc[valid, "Uncorrected_Yield"])
#     / np.log(df.loc[valid, "HA"] / df.loc[valid, "SHAPE_Area"])
# )

# # =========================
# # Global alpha using regression through origin
# # log(Corrected/Uncorrected) = (alpha - 1) * log(HA/SHAPE_Area)
# # =========================
# x = np.log(df.loc[valid, "HA"] / df.loc[valid, "SHAPE_Area"]).values.reshape(-1, 1)
# y = np.log(df.loc[valid, "Corrected_Yield"] / df.loc[valid, "Uncorrected_Yield"]).values

# model = LinearRegression(fit_intercept=False)
# model.fit(x, y)

# beta = model.coef_[0]
# alpha_global = 1 + beta

# # Add global alpha as a constant column
# df["Alpha_Global"] = alpha_global

# # =========================
# # Use global alpha to estimate empirical yield
# # =========================
# valid_emp = (
#     df["Uncorrected_Yield"].notna() &
#     df["SHAPE_Area"].notna() &
#     df["HA"].notna() &
#     (df["Uncorrected_Yield"] > 0) &
#     (df["SHAPE_Area"] > 0) &
#     (df["HA"] > 0)
# )

# df["Yield_Empirical_GlobalAlpha"] = np.nan
# df.loc[valid_emp, "Yield_Empirical_GlobalAlpha"] = (
#     df.loc[valid_emp, "Uncorrected_Yield"] *
#     (df.loc[valid_emp, "HA"] / df.loc[valid_emp, "SHAPE_Area"]) ** (alpha_global - 1)
# )

# # =========================
# # Use row-wise alpha to reconstruct yield
# # =========================
# df["Yield_Empirical_RowAlpha"] = np.nan
# row_valid = valid.copy()

# df.loc[row_valid, "Yield_Empirical_RowAlpha"] = (
#     df.loc[row_valid, "Uncorrected_Yield"] *
#     (df.loc[row_valid, "HA"] / df.loc[row_valid, "SHAPE_Area"]) ** (df.loc[row_valid, "Alpha_Row"] - 1)
# )

# # =========================
# # Summary
# # =========================
# print("Global alpha =", alpha_global)
# print("Median row-wise alpha =", df["Alpha_Row"].median())
# print("Mean row-wise alpha =", df["Alpha_Row"].mean())

# # Save file
# df.to_excel(output_path, index=False)
# print(f"Saved to: {output_path}")

In [ ]:
#nonlinear_model

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_XY_Final_Fixed.xlsx"
#file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
output_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_nonlinear_model.xlsx"

df = pd.read_excel(file_path)

# convert numeric columns
cols = ["Uncorrected_Yield","Corrected_Yield","SHAPE_Area","HA","Join_Count"]
for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# compute ratios
df["area_ratio"] = df["HA"] / df["SHAPE_Area"]
df["stand_ratio"] = df["Join_Count"] / 150

valid = (
    df["Uncorrected_Yield"] > 0
) & (
    df["Corrected_Yield"] > 0
) & (
    df["area_ratio"] > 0
) & (
    df["stand_ratio"] > 0
)

# log transform
X1 = np.log(df.loc[valid,"area_ratio"])
X2 = np.log(df.loc[valid,"stand_ratio"])

X = np.column_stack([X1,X2])

y = np.log(df.loc[valid,"Corrected_Yield"] / df.loc[valid,"Uncorrected_Yield"])

model = LinearRegression()
model.fit(X,y)

alpha = model.coef_[0]
beta = model.coef_[1]

print("Estimated alpha:",alpha)
print("Estimated beta:",beta)

# compute predicted yield
df["Yield_Nonlinear_Model"] = (
    df["Uncorrected_Yield"]
    * (df["area_ratio"] ** alpha)
    * (df["stand_ratio"] ** beta)
)

df.to_excel(output_path,index=False)

print("File saved:",output_path)

Estimated alpha: -0.0038701762026802674
Estimated beta: -0.2025503323870862
File saved: C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_nonlinear_model.xlsx


In [3]:
import pandas as pd
import numpy as np
file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_XY_Final_Fixed.xlsx"
#file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
output_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_NewYield.xlsx"

df = pd.read_excel(file_path)

# Convert numeric columns
cols = ["Uncorrected_Yield","SHAPE_Area","HA","Join_Count"]
for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Example parameters (replace with your estimated values)
alpha = 0.063
beta = -0.187

# Compute ratios
df["Area_Ratio"] = df["HA"] / df["SHAPE_Area"]
df["Stand_Ratio"] = df["Join_Count"] / 150

# New corrected yield column
df["Yield_Area_Stand_Corrected"] = (
    df["Uncorrected_Yield"]
    * (df["Area_Ratio"] ** alpha)
    * (df["Stand_Ratio"] ** beta)
)

# Save dataset
df.to_excel(output_path, index=False)

print("New yield column created and file saved.")

New yield column created and file saved.


In [ ]:
#Because crop plants partially compensate for missing neighbors, yield does not scale linearly 
#with plant density. Therefore, stand density was incorporated using a power-law 
#compensation term, where the exponent β was estimated empirically from the dataset.
#-----------------------------------------------Start--------------------------------------------

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"

df = pd.read_excel(file_path)

# Convert columns
cols = ["PLOTWT", "HA", "Join_Count", "Uncorrected_Yield"]
for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Yield from mass
df["Yield_mass"] = (df["PLOTWT"] / 1000) / df["HA"] * 10000

# Stand ratio
df["Stand_ratio"] = df["Join_Count"] / 150

# Ratio used in y
df["Yield_ratio"] = df["Yield_mass"] / df["Uncorrected_Yield"]

# Strict valid filter
valid = (
    df["Yield_mass"].notna() &
    df["Stand_ratio"].notna() &
    df["Uncorrected_Yield"].notna() &
    df["Yield_ratio"].notna() &
    np.isfinite(df["Yield_mass"]) &
    np.isfinite(df["Stand_ratio"]) &
    np.isfinite(df["Uncorrected_Yield"]) &
    np.isfinite(df["Yield_ratio"]) &
    (df["Yield_mass"] > 0) &
    (df["Stand_ratio"] > 0) &
    (df["Uncorrected_Yield"] > 0) &
    (df["Yield_ratio"] > 0)
)

# Build X and y
X = np.log(df.loc[valid, "Stand_ratio"]).values.reshape(-1, 1)
y = np.log(df.loc[valid, "Yield_ratio"]).values

# Fit model
model = LinearRegression()
model.fit(X, y)

beta = model.coef_[0]

print("Estimated compensation parameter β =", beta)

# New corrected yield column
df["Yield_compensation_corrected"] = np.nan
df.loc[valid, "Yield_compensation_corrected"] = (
    df.loc[valid, "Yield_mass"] *
    (df.loc[valid, "Stand_ratio"] ** beta)
)

print("Number of valid rows used:", valid.sum())
print("Number of removed rows:", len(df) - valid.sum())

Estimated compensation parameter β = -0.401210699823101
Number of valid rows used: 4511
Number of removed rows: 1


In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# -----------------------------
# File paths
# -----------------------------
file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_XY_Final_Fixed.xlsx"
#file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
output_file = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_compensation_yield.xlsx"

# -----------------------------
# Read dataset
# -----------------------------
df = pd.read_excel(file_path)

# Ensure numeric
cols = ["PLOTWT","HA","Join_Count","Uncorrected_Yield"]
for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# -----------------------------
# Compute yield from biomass
# -----------------------------
df["Yield_mass"] = (df["PLOTWT"] / 1000) / df["HA"] * 10000

# Stand ratio
df["Stand_ratio"] = df["Join_Count"] / 150

# Ratio used for model
df["Yield_ratio"] = df["Yield_mass"] / df["Uncorrected_Yield"]

# -----------------------------
# Valid rows for regression
# -----------------------------
valid = (
    df["Yield_mass"].notna() &
    df["Stand_ratio"].notna() &
    df["Uncorrected_Yield"].notna() &
    (df["Yield_mass"] > 0) &
    (df["Stand_ratio"] > 0) &
    (df["Uncorrected_Yield"] > 0)
)

# -----------------------------
# Estimate compensation parameter β
# -----------------------------
X = np.log(df.loc[valid, "Stand_ratio"]).values.reshape(-1,1)
y = np.log(df.loc[valid, "Yield_ratio"]).values

model = LinearRegression()
model.fit(X, y)

beta = model.coef_[0]

print("Estimated compensation parameter β =", beta)

# -----------------------------
# Compute corrected yield column
# -----------------------------
df["Yield_compensation_corrected"] = (
    df["Yield_mass"] *
    (df["Stand_ratio"] ** beta)
)

# -----------------------------
# Save new dataset
# -----------------------------
df.to_excel(output_file, index=False)

print("New file saved at:")
print(output_file)

Estimated compensation parameter β = -0.43396660853683267
New file saved at:
C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_compensation_yield.xlsx


In [ ]:
#--------------------------------------End----------------------------------------------------

In [ ]:
#Rectangular hyperbola stand-compensation model

In [5]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_XY_Final_Fixed.xlsx"
#file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
output_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_recthyperbola_area_stand.xlsx"

N_OLD = 150

df = pd.read_excel(file_path)

for col in ["PLOTWT", "HA", "SHAPE_Area", "Join_Count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Yield_mass_kg_ha"] = ((df["PLOTWT"] / 1000.0) / df["HA"]) * 10000.0
df["Stand_Ratio"] = df["Join_Count"] / N_OLD
df["Area_Ratio"] = df["HA"] / df["SHAPE_Area"]

valid = (
    df["Yield_mass_kg_ha"].notna() &
    df["Stand_Ratio"].notna() &
    df["Area_Ratio"].notna() &
    (df["Yield_mass_kg_ha"] > 0) &
    (df["Stand_Ratio"] > 0) &
    (df["Area_Ratio"] > 0)
)

fit_df = df.loc[valid].copy()

def rect_hyperbola_area_stand(X, Ymax, c, alpha):
    D, A = X
    return Ymax * ((c * D) / (1.0 + c * D)) * (A ** alpha)

xdata = np.vstack([
    fit_df["Stand_Ratio"].values,
    fit_df["Area_Ratio"].values
])
ydata = fit_df["Yield_mass_kg_ha"].values

p0 = [np.percentile(ydata, 95), 2.0, 0.5]
bounds = ([0, 0, -5], [np.inf, np.inf, 5])

params, _ = curve_fit(
    rect_hyperbola_area_stand,
    xdata,
    ydata,
    p0=p0,
    bounds=bounds,
    maxfev=50000
)

Ymax_est, c_est, alpha_est = params

print("Ymax =", Ymax_est)
print("c =", c_est)
print("alpha =", alpha_est)

df["Yield_RectHyperbola_AreaStand_Corrected"] = np.nan
pred_valid = valid.copy()

xpred = np.vstack([
    df.loc[pred_valid, "Stand_Ratio"].values,
    df.loc[pred_valid, "Area_Ratio"].values
])

df.loc[pred_valid, "Yield_RectHyperbola_AreaStand_Corrected"] = rect_hyperbola_area_stand(
    xpred, Ymax_est, c_est, alpha_est
)

df.to_excel(output_path, index=False)
print("Saved to:", output_path)

Ymax = 2212.670487827166
c = 82.5647741001672
alpha = -1.060209301676625
Saved to: C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_recthyperbola_area_stand.xlsx


In [ ]:
#Spatial adjustment models

In [16]:
# import pandas as pd
# import numpy as np
# import statsmodels.api as sm
# from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# # =========================================
# # File paths
# # =========================================
# file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
# output_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_spatial_adjusted_yield.xlsx"

# # =========================================
# # User settings
# # Replace these with your actual column names
# # =========================================
# ROW_COL_NAME = "Row"
# COL_COL_NAME = "Column"
# N_OLD = 150

# # =========================================
# # Read data
# # =========================================
# df = pd.read_excel(file_path)

# # =========================================
# # Required columns
# # =========================================
# required_cols = ["PLOTWT", "HA", "SHAPE_Area", "Join_Count", ROW_COL_NAME, COL_COL_NAME]
# missing_cols = [c for c in required_cols if c not in df.columns]
# if missing_cols:
#     raise ValueError(f"Missing required columns: {missing_cols}")

# # Convert numeric columns
# for col in required_cols:
#     df[col] = pd.to_numeric(df[col], errors="coerce")

# # =========================================
# # Step 1: Calculate base yield from mass
# # =========================================
# df["Yield_mass_kg_ha"] = ((df["PLOTWT"] / 1000.0) / df["HA"]) * 10000.0

# # Biological ratios
# df["Stand_Ratio"] = df["Join_Count"] / N_OLD
# df["Area_Ratio"] = df["HA"] / df["SHAPE_Area"]

# # Spatial polynomial terms
# df["Row2"] = df[ROW_COL_NAME] ** 2
# df["Col2"] = df[COL_COL_NAME] ** 2
# df["Row_Col"] = df[ROW_COL_NAME] * df[COL_COL_NAME]

# # =========================================
# # Step 2: Valid rows
# # =========================================
# valid = (
#     df["Yield_mass_kg_ha"].notna() &
#     df["Stand_Ratio"].notna() &
#     df["Area_Ratio"].notna() &
#     df[ROW_COL_NAME].notna() &
#     df[COL_COL_NAME].notna() &
#     np.isfinite(df["Yield_mass_kg_ha"]) &
#     np.isfinite(df["Stand_Ratio"]) &
#     np.isfinite(df["Area_Ratio"]) &
#     np.isfinite(df[ROW_COL_NAME]) &
#     np.isfinite(df[COL_COL_NAME]) &
#     (df["Yield_mass_kg_ha"] > 0) &
#     (df["Stand_Ratio"] > 0) &
#     (df["Area_Ratio"] > 0)
# )

# fit_df = df.loc[valid].copy()

# if len(fit_df) < 10:
#     raise ValueError("Not enough valid rows to fit the spatial adjustment model.")

# # =========================================
# # Step 3: Fit spatial adjustment model
# # log(yield) ~ log(stand_ratio) + log(area_ratio) + row + col + row² + col² + row*col
# # =========================================
# fit_df["log_y"] = np.log(fit_df["Yield_mass_kg_ha"])
# fit_df["log_stand"] = np.log(fit_df["Stand_Ratio"])
# fit_df["log_area"] = np.log(fit_df["Area_Ratio"])

# X = fit_df[["log_stand", "log_area", ROW_COL_NAME, COL_COL_NAME, "Row2", "Col2", "Row_Col"]]
# X = sm.add_constant(X)
# y = fit_df["log_y"]

# model = sm.OLS(y, X).fit()

# print(model.summary())

# # =========================================
# # Step 4: Predict adjusted yield for all valid rows
# # =========================================
# df["Yield_Spatial_Adjusted"] = np.nan

# pred_df = df.loc[valid].copy()
# pred_df["log_stand"] = np.log(pred_df["Stand_Ratio"])
# pred_df["log_area"] = np.log(pred_df["Area_Ratio"])

# X_pred = pred_df[["log_stand", "log_area", ROW_COL_NAME, COL_COL_NAME, "Row2", "Col2", "Row_Col"]]
# X_pred = sm.add_constant(X_pred, has_constant="add")

# pred_log_y = model.predict(X_pred)

# # Back-transform from log scale
# df.loc[valid, "Yield_Spatial_Adjusted"] = np.exp(pred_log_y)

# # =========================================
# # Step 5: Diagnostics
# # =========================================
# eval_df = df.loc[valid & df["Yield_Spatial_Adjusted"].notna()].copy()

# r2 = r2_score(eval_df["Yield_mass_kg_ha"], eval_df["Yield_Spatial_Adjusted"])
# rmse = np.sqrt(mean_squared_error(eval_df["Yield_mass_kg_ha"], eval_df["Yield_Spatial_Adjusted"]))
# mae = mean_absolute_error(eval_df["Yield_mass_kg_ha"], eval_df["Yield_Spatial_Adjusted"])

# print(f"\nModel performance against mass-based yield:")
# print(f"R2   = {r2:.4f}")
# print(f"RMSE = {rmse:.4f}")
# print(f"MAE  = {mae:.4f}")

# # =========================================
# # Save new file
# # =========================================
# df.to_excel(output_path, index=False)

# print("\nNew file saved to:")
# print(output_path)